[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/un1tz3r0/vfbd3/blob/main/notebooks/vfbd3_colab.ipynb)

# vfbd3 — restyle video with diffusion

Two engines:

- **vace** — the Wan VACE video model, run over the clip in overlapping windows. Stable, coherent motion.
- **frame** — per-frame img2img (SDXL, Flux or SD3) with motion-compensated feedback of the previous output (the original *controlnetvideo* technique). Trails, drift and all.

Runtime → Change runtime type → **T4 GPU** (or better). Output frames are kept in `<output>.work/`, so if the session disconnects, re-running the same cell continues where it stopped — put the output on Drive to survive a full runtime reset.

Flux and SD3 are gated on Hugging Face: accept their licences on the model pages, then add an `HF_TOKEN` secret (key icon in the left bar) with notebook access.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "vfbd3[upscale,aux,flux,youtube] @ git+https://github.com/un1tz3r0/vfbd3"
# Colab's preinstalled torchao is too old for peft, which then refuses to load any LoRA; vfbd3 doesn't use it
!pip uninstall -q -y torchao

In [ ]:
#@title Storage and input video
use_drive = True  #@param {type:"boolean"}
project_dir = "vfbd3"  #@param {type:"string"}
input_video = "https://www.youtube.com/watch?v=aqz-KE-bpKQ"  #@param {type:"string"}
max_height = 1080  #@param [480, 720, 1080] {type:"raw"}
#@markdown `input_video` may be a YouTube (or any yt-dlp) link, downloaded once into the project folder; a path (relative to the project folder, or absolute); or empty, to upload a file. If YouTube refuses the download ("sign in to confirm you're not a bot"), upload the file instead.
import os
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    base = f"/content/drive/MyDrive/{project_dir}"
else:
    base = f"/content/{project_dir}"
os.makedirs(base, exist_ok=True)
os.chdir(base)
if not input_video:
    from google.colab import files
    input_video = next(iter(files.upload()))
elif input_video.startswith(("http://", "https://")):
    from vfbd3 import video
    input_video = video.download(input_video, base, max_height=max_height)
input_video = os.path.abspath(input_video)
print("input:", input_video, "\nworking in:", base)

## Engine 1: vace (video model)

Generates at about 480p / 16 fps and upscales afterwards. On a T4 expect several minutes per 81-frame window (about 5 seconds of video).

- `anchor = first` passes the first output frame to every window as a reference; try `none` if the first frame's composition bleeds into later ones.
- `warp_frames > 0` is experimental: it pre-fills that many frames after each overlap by flow-warping the last finished frame.

In [ ]:
#@title Run vace
output_video = "out_vace.mp4"  #@param {type:"string"}
prompt = "a colorful murakami mural, superflat, smiling flowers"  #@param {type:"string"}
control = "depth"  #@param ["depth", "canny", "gray", "none"]
control_scale = 1.0  #@param {type:"number"}
fast = "causvid"  #@param ["none", "causvid", "selfforcing"]
steps = 0  #@param {type:"integer"}
guidance = 0  #@param {type:"number"}
chunk_frames = 81  #@param [49, 65, 81] {type:"raw"}
overlap = 9  #@param [5, 9, 13] {type:"raw"}
anchor = "first"  #@param ["first", "none"]
warp_frames = 0  #@param {type:"integer"}
upscale_to = 720  #@param [0, 720, 1080, 1440] {type:"raw"}
start = 0  #@param {type:"number"}
duration = 0  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
restart = False  #@param {type:"boolean"}
#@markdown `fast` loads a step-distillation LoRA; `steps = 0` and `guidance = 0` use the preset's values (30 and 5.0 without one). `duration = 0` processes the whole clip. Tick `restart` to throw away a previous partial run of this output.
from vfbd3 import vace, common
cfg = vace.VaceConfig(prompt=prompt, control=control, control_scale=control_scale, fast=None if fast == "none" else fast,
    steps=steps or None, guidance=guidance or None,
    chunk_frames=chunk_frames, overlap=overlap, anchor=anchor, warp_frames=warp_frames,
    upscale=upscale_to or None, start=start or None, duration=duration or None, seed=seed, restart=restart)
vace.run(input_video, output_video, cfg)
common.free_memory()

## Engine 2: frame (feedback loop)

Generates at 720p directly. `denoise` is how much of each frame is re-imagined; `feedthrough` mixes the raw input back into the loop (0 = pure feedback).

**Backends.** `sdxl` (any SDXL checkpoint as `model`, few-step `fast` presets, reference attention). `flux`: FLUX.1-dev with a 4-bit GGUF transformer and ControlNet-Union-Pro-2.0, about 11 GB on the T4; `fast = schnell` for 4 steps. `sd3`: SD3 Medium with the InstantX/alimama ControlNets (depth, canny/lineart, pose, hed/pidi).

**Reference attention** (sdxl) is controlnetvideo's reference-only mode: every self-attention layer also attends to the previous output (`reference`) and/or the current input (`reference_input`). The old `refxl` recipe is `feedthrough = 1`, `denoise ≈ 0.5`, `reference = 1`, `reference_adain = 0.5`: the input is re-noised every frame, the previous output supplies the look.

**Colour creep.** A feedback loop compounds whatever the model and autoencoder do to colour every frame. The tiny autoencoder's decoder alone nearly doubles saturation over 15 round trips, so `tiny_vae = encoder` keeps the full decoder. To hold the look steady, use `color_fix` or one of the feedback filters below.

**Live preview panels** (`live_preview`, comma-separated, or `all`). `input`, `control`, `reference`, `warped`, `feedback`, `init`, `output`: the loop's images. `occlusion`: newly revealed areas. `noise`: the starting noise. Motion and attention:

- `flow`: the motion the previous output was moved by, as colour (hue = direction, brightness = speed).
- `vectors`: arrows over the previous input, scaled to fit a cell. `attention` (sdxl): confidence-weighted correspondence arrows over the input. `attended` (sdxl): the input resampled through that correspondence.
- `flow_vectors`: the same motion as a grid of arrows over the previous and current input frames, blended 50/50. Each arrow starts where a point was and ends where it is now, so when the motion is right its head sits on the matching feature.
- `correspondence` (sdxl): where each part of the output was drawn from in the input, as arrows over the two blended; dim arrows are low confidence. `confidence` shows that confidence alone.
- `attention_split` (sdxl): how attention divides between the frame itself (blue), the previous output (orange) and the input (green). Needs `reference` or `reference_input` above 0.

`attention`, `attended`, `correspondence` and `confidence` add a reference pass on the input even at `reference_input = 0` (the output is unchanged, but each step costs one more UNet pass), and so does `all` on sdxl.

In [ ]:
#@title Run frame
output_video = "out_frame.mp4"  #@param {type:"string"}
prompt = "murakami mural"  #@param {type:"string"}
negative_prompt = ""  #@param {type:"string"}
backend = "sdxl"  #@param ["sdxl", "flux", "sd3"]
model = ""  #@param {type:"string"}
control = "depth"  #@param ["depth", "depth_large", "zoe", "midas", "normal", "normalbae", "canny", "lineart", "lineart_anime", "mlsd", "hed", "pidi", "pose"]
control_scale = 1.0  #@param {type:"number"}
short_side = 720  #@param {type:"integer"}
fps = 0  #@param {type:"number"}
fast = "lcm"  #@param ["none", "lcm", "lightning", "dmd2", "turbo", "schnell"]
tiny_vae = "encoder"  #@param ["none", "encoder", "both"]
steps = 0  #@param {type:"integer"}
guidance = 0  #@param {type:"number"}
denoise = 0.5  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
feedthrough = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown ---
reference = 0.0  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
reference_warp = True  #@param {type:"boolean"}
reference_input = 0.0  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
reference_adain = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
flow_remap = 0.0  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
ip_adapter = 0.0  #@param {type:"number"}
#@markdown ---
motion_estimator = "dis"  #@param ["dis", "farneback", "raft", "raft_small"]
occlusion = True  #@param {type:"boolean"}
occlusion_denoise = 1.0  #@param {type:"slider", min:0.05, max:1.0, step:0.05}
noise = "warp"  #@param ["warp", "fixed", "random"]
seed = 0  #@param {type:"integer"}
color_fix = 0.2  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
feedback_filter = "colorlock amount=0.5"  #@param ["", "colorlock amount=0.5", "colorlock amount=0.8, sat=0.9", "autolevel amount=0.5", "equalize clip=1.5, amount=0.3", "colorlock amount=0.5; tdenoise alpha=0.3", "zoom rate=0.01; hueshift deg=2; colorlock amount=0.6"] {allow-input: true}
output_filter = ""  #@param {type:"string"}
#@markdown ---
live_preview = "input,flow,occlusion,output"  #@param ["none", "output", "input,flow,occlusion,output", "input,flow_vectors,occlusion,output", "input,flow_vectors,correspondence,attention_split,output", "reference,warped,vectors,output", "input,vectors,attention,attended,confidence,output", "all"] {allow-input: true}
preview_file = ""  #@param {type:"string"}
upscale_to = 0  #@param [0, 1080, 1440, 2160] {type:"raw"}
start = 0  #@param {type:"number"}
duration = 0  #@param {type:"number"}
restart = False  #@param {type:"boolean"}
#@markdown Leave `model` empty for the backend's default. `fast` picks a few-step setup (lcm/lightning/dmd2/turbo for sdxl, schnell for flux, none for sd3); `steps = 0` and `guidance = 0` use its values, and a frame runs `steps * denoise` steps. `reference*` and `flow_remap` are sdxl only; `flow_remap` needs a reference pass and adds one even at `reference_input = 0`. `live_preview` shows a mosaic under this cell as frames finish (panels: see the cell above); `preview_file` also writes it to that path (atomically) for an auto-refreshing viewer. Filter chains: `!vfbd3 filters` lists them.
from IPython.display import display
from PIL import Image
from vfbd3 import frame, common
none = lambda v: None if v in ("none", "", 0) else v
handle = display(Image.new("RGB", (16, 16)), display_id=True) if none(live_preview) else None
cfg = frame.FrameConfig(prompt=prompt, negative_prompt=negative_prompt, backend=backend, model=none(model), control=control,
    control_scale=control_scale, short_side=short_side, fps=none(fps), fast=none(fast), tiny_vae=none(tiny_vae),
    steps=none(steps), guidance=none(guidance), denoise=denoise, feedthrough=feedthrough,
    reference=reference, reference_warp=reference_warp, reference_input=reference_input, reference_adain=reference_adain,
    flow_remap=flow_remap, ip_adapter=ip_adapter, motion_estimator=motion_estimator, occlusion=occlusion,
    occlusion_denoise=occlusion_denoise, noise=noise, seed=seed, color_fix=color_fix,
    feedback_filter=none(feedback_filter), output_filter=none(output_filter),
    preview=none(preview_file), preview_panels=none(live_preview) or "output",
    upscale=none(upscale_to), start=none(start), duration=none(duration), restart=restart)
frame.run(input_video, output_video, cfg, on_preview=(lambda image: handle.update(Image.fromarray(image))) if handle else None)
common.free_memory()

In [ ]:
#@title Preview the last output
from IPython.display import Video
Video(output_video, embed=True, width=720)